In [1]:
import pandas as pd
import numpy as np
import itertools

#import NLP preprocessing libraries
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer

#import test_train_split
from sklearn.model_selection import train_test_split

#import classification algorthms -- just two for trying. But we can try with n number of algorithms
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.naive_bayes import MultinomialNB

#evaluation metrics
from sklearn import metrics

import matplotlib.pyplot as plt

In [2]:
#read daatset
df = pd.read_csv('news.csv', index_col=None)
df

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL
...,...,...,...,...
6330,4490,State Department says it can't find emails fro...,The State Department told the Republican Natio...,REAL
6331,8062,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,FAKE
6332,8622,Anti-Trump Protesters Are Tools of the Oligarc...,Anti-Trump Protesters Are Tools of the Oligar...,FAKE
6333,4021,"In Ethiopia, Obama seeks progress on peace, se...","ADDIS ABABA, Ethiopia —President Obama convene...",REAL


In [3]:
dataset = df.drop("Unnamed: 0", axis=1) #axis=o means row, axis=1 means column
#dataset

In [4]:
#imput and output

x = dataset["text"]
y = dataset["label"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=53)

In [7]:
#preprocessing for NLP -- text to numbers or vectors. This is called feature extraction in NLP

tfidf_vectorizer = TfidfVectorizer(stop_words='english') #stop words means non-important repeated words in english like is, was so etc..
tfidf_train = tfidf_vectorizer.fit_transform(X_train)
tfidf_test = tfidf_vectorizer.transform(X_test)

#Preprocessing output is optional. Bcz output label (REAL / FAKE) does not need to be converted into 0 and 1 because the MultinomialNB classifier can directly accept string labels
#So label encoding is skipped here.

In [9]:
#1) let's use one MultinomialNB classification algorithm for cerating model now.
classifier = MultinomialNB()
classifier.fit(tfidf_train, y_train)

MultinomialNB()

In [10]:
pred = classifier.predict(tfidf_test)

In [11]:
score = metrics.accuracy_score(y_test,pred)
print("accuracy: %0.3f" % score)

accuracy: 0.857


In [12]:
cm = metrics.confusion_matrix(y_test, pred, labels=['FAKE', 'REAL'])
cm

array([[ 739,  269],
       [  31, 1052]], dtype=int64)

In [13]:
#CLASSIFICATION REPORT
from sklearn.metrics import classification_report
report = classification_report (y_test,pred)
print(report)

              precision    recall  f1-score   support

        FAKE       0.96      0.73      0.83      1008
        REAL       0.80      0.97      0.88      1083

    accuracy                           0.86      2091
   macro avg       0.88      0.85      0.85      2091
weighted avg       0.88      0.86      0.85      2091



In [14]:
# test for real input case.
dataset['text'][0]


'Daniel Greenfield, a Shillman Journalism Fellow at the Freedom Center, is a New York writer focusing on radical Islam. \nIn the final stretch of the election, Hillary Rodham Clinton has gone to war with the FBI. \nThe word “unprecedented” has been thrown around so often this election that it ought to be retired. But it’s still unprecedented for the nominee of a major political party to go war with the FBI. \nBut that’s exactly what Hillary and her people have done. Coma patients just waking up now and watching an hour of CNN from their hospital beds would assume that FBI Director James Comey is Hillary’s opponent in this election. \nThe FBI is under attack by everyone from Obama to CNN. Hillary’s people have circulated a letter attacking Comey. There are currently more media hit pieces lambasting him than targeting Trump. It wouldn’t be too surprising if the Clintons or their allies were to start running attack ads against the FBI. \nThe FBI’s leadership is being warned that the entir

In [15]:
X_train[[0]]

0    Daniel Greenfield, a Shillman Journalism Fello...
Name: text, dtype: object

In [16]:
tfidf_train[[0]]

<1x56922 sparse matrix of type '<class 'numpy.float64'>'
	with 0 stored elements in Compressed Sparse Row format>

In [17]:
classifier.predict(tfidf_train[[0]])

array(['FAKE'], dtype='<U4')

In [16]:
#If we need to deploy as stand alone model, then we need to save the tfidf vector preprocess mode and the classification model as well
#Then Input needs to be passed through vectorization and then classification model.


In [18]:
#2) Let's try to create model using PassiveAggresive Classifier

clf = PassiveAggressiveClassifier(max_iter=1000, random_state=0, tol=1e-3)
clf.fit(tfidf_train, y_train)

PassiveAggressiveClassifier(random_state=0)

In [19]:
y_pred = clf.predict(tfidf_test)

In [20]:
#CLASSIFICATION REPORT
clf_report = classification_report (y_test,y_pred)
print(clf_report)

              precision    recall  f1-score   support

        FAKE       0.92      0.95      0.93      1008
        REAL       0.95      0.92      0.94      1083

    accuracy                           0.93      2091
   macro avg       0.93      0.93      0.93      2091
weighted avg       0.93      0.93      0.93      2091



In [27]:
acc_score = metrics.accuracy_score(y_test,y_pred)
print("accuracy: %0.3f" % acc_score)

accuracy: 0.934


In [22]:
#y[2]

In [23]:
#test the model.
clf.predict(tfidf_train[[2]])

array(['REAL'], dtype='<U4')

In [24]:
#Since clf report is good for passive aggressive , we can save that models for deployment.

In [25]:
#Save the vectorizer for input preprocessing

import joblib
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']

In [26]:
#Save the model

joblib.dump(clf, 'model.pkl')

['model.pkl']